In [7]:
%%sql
CREATE OR REPLACE VIEW filtered_1m AS
WITH init_ranked AS (
    SELECT
        *,
        ts_event::TIMESTAMP::DATE AS tick_date,
        row_number() OVER (PARTITION BY symbol, ts_event::TIMESTAMP::DATE ORDER BY ts_event ASC) AS rank_asc,
        count(*) OVER (PARTITION BY symbol, ts_event::TIMESTAMP::DATE) AS day_count
    FROM es_futures
),
fully_ranked AS (
    SELECT
        *,
        day_count - rank_asc + 1 AS rank_desc,
        high - low AS hl_range

    FROM init_ranked
),

rolling_stats AS (
    SELECT
        *,
        CASE
            WHEN rank_asc <= 30 THEN median(volume) FILTER(WHERE rank_asc <= 60) OVER w_day
            WHEN rank_desc <= 30 THEN median(volume) FILTER(WHERE rank_desc <= 60) OVER w_day
            ELSE median(volume) OVER w_roll
        END AS median_vol,

        CASE
            WHEN rank_asc <= 30 THEN mad(volume) FILTER(WHERE rank_asc <= 60) OVER w_day
            WHEN rank_desc <= 30 THEN mad(volume) FILTER(WHERE rank_desc <= 60) OVER w_day
            ELSE mad(volume) OVER w_roll
        END AS mad_vol,

        CASE
            WHEN rank_asc <= 30 THEN median("close") FILTER(WHERE rank_asc <= 60) OVER w_day
            WHEN rank_desc <= 30 THEN median("close") FILTER(WHERE rank_desc <= 60) OVER w_day
            ELSE median("close") OVER w_roll
        END AS median_close,

        CASE
            WHEN rank_asc <= 30 THEN mad("close") FILTER(WHERE rank_asc <= 60) OVER w_day
            WHEN rank_desc <= 30 THEN mad("close") FILTER(WHERE rank_desc <= 60) OVER w_day
            ELSE mad("close") OVER w_roll
        END AS mad_close,

        CASE
            WHEN rank_asc <= 30 THEN median(hl_range) FILTER(WHERE rank_asc <= 60) OVER w_day
            WHEN rank_desc <= 30 THEN median(hl_range) FILTER(WHERE rank_desc <= 60) OVER w_day
            ELSE median(hl_range) OVER w_roll
        END AS median_hl,

        CASE
            WHEN rank_asc <= 30 THEN mad(hl_range) FILTER(WHERE rank_asc <= 60) OVER w_day
            WHEN rank_desc <= 30 THEN mad(hl_range) FILTER(WHERE rank_desc <= 60) OVER w_day
            ELSE mad(hl_range) OVER w_roll
        END AS mad_hl

    FROM fully_ranked
    WINDOW
        w_day AS (PARTITION BY symbol, tick_date),
        w_roll AS (PARTITION BY symbol, tick_date ORDER BY ts_event ROWS BETWEEN 30 PRECEDING AND 30 FOLLOWING)
),

check_valid AS (
    SELECT
        *,
        (abs("close" - median_close) <= 3 * mad_close + 0.01)
        OR
        (abs(hl_range - median_hl) <= 3 * mad_hl + 0.01)
            AS is_valid_close,

        abs(volume - median_vol) <= 3 * mad_vol + 0.01 AS is_valid_vol

    FROM rolling_stats
),

fill_forward_prep AS (
    SELECT
        *,

        LAST_VALUE(IF(is_valid_close, "close", NULL) IGNORE NULLS)
            OVER ( ORDER BY ts_event ASC)
        AS last_valid_close

    FROM check_valid
)
SELECT
    * EXCLUDE("open", high, low, "close", volume, is_valid_close, is_valid_vol,
    median_vol, median_close, median_hl, mad_vol, mad_close, mad_hl, hl_range, last_valid_close),
    tick_date,
    UNNEST(
        IF(
            is_valid_close,
            {'"open"': "open", 'high': high, 'low': low, '"close"': "close", 'volume': IF(is_valid_vol, volume, median_vol)},
            {'"open"': last_valid_close, 'high': last_valid_close, 'low': last_valid_close, '"close"': last_valid_close, 'volume': 0}
            )
    )
FROM fill_forward_prep;

,tick_date,ts_event,"""open""",high,low,"""close""",volume,CASE WHEN (is_valid_vol) THEN (volume) ELSE median_vol END
0,2023-10-01,2023-10-01T22:00Z,4349.00,4354.00,4344.25,4352.75,3052,187.0
1,2023-10-01,2023-10-01T22:00Z,4445.00,4445.00,4445.00,4445.00,0,151.0
2,2023-10-01,2023-10-01T22:00Z,48.05,48.05,48.05,48.05,11,11.0
3,2023-10-01,2023-10-01T22:00Z,4445.00,4445.00,4445.00,4445.00,1,1.0
4,2023-10-01,2023-10-01T22:01Z,48.10,48.10,48.10,48.10,1,1.0
...,...,...,...,...,...,...,...,...
1697663,2026-10-01,2026-09-30T23:55Z,7744.25,7744.75,7744.00,7744.00,85,85.0
1697664,2026-10-01,2026-09-30T23:56Z,7744.00,7744.00,7743.25,7743.50,193,193.0
1697665,2026-10-01,2026-09-30T23:57Z,7743.25,7743.75,7743.25,7743.50,62,62.0
1697666,2026-10-01,2026-09-30T23:58Z,7743.25,7743.75,7743.25,7743.75,123,123.0


FLOW:
- Adjust cme day
- [[Bin unnecessary instruments]]
- (FIND ROLLOVER DATES?)
- Handle rollovers
- Pass BG over NON-AGGREGATED DATA
- Coarse grain
- Calculate Intraday volatility, returns
- GGWP

In [ ]:
%%sql
CREATE OR REPLACE VIEW cme_es_futures AS
       SELECT
           * EXCLUDE(ts_event),
       CASE
            WHEN date_part('hour', ts_event AT TIME ZONE 'America/Chicago') >= 17
            THEN (date_trunc('day', ts_event AT TIME ZONE 'America/Chicago') + INTERVAL 1 DAY)::DATE
            ELSE date_trunc('day', ts_event AT TIME ZONE 'America/Chicago')::DATE
       END AS cme_trade_date,
           (ts_event AT TIME ZONE 'America/Chicago')::TIME AS cme_ts_event
FROM es_futures;

In [19]:
%%sql
CREATE OR REPLACE VIEW cme_rollover_months AS
WITH
rollover_month_volumes AS (
    SELECT
        cme_trade_date,
        symbol,
        sum(volume) AS daily_vol
    FROM cme_es_futures
    WHERE symbol NOT LIKE '%-%' -- excl calendar spread
      -- ES rollover months (H, M, U, Z)
      AND date_part('month', cme_trade_date) IN (3, 6, 9, 12)
    GROUP BY cme_trade_date, symbol
),

top_vol_pair_daily AS (
    SELECT
        cme_trade_date,
        arg_max(symbol, daily_vol, 2) AS top_2_vol
    FROM rollover_month_volumes
    GROUP BY cme_trade_date
    ORDER BY cme_trade_date DESC
)

SELECT
    cme_trade_date AS rollover_date,
    LAG(top_2_vol[1]) OVER (ORDER BY cme_trade_date ASC) AS outgoing_contract,
    top_2_vol[1] AS incoming_contract,
    top_2_vol AS new_order
FROM top_vol_pair_daily
QUALIFY incoming_contract != outgoing_contract
ORDER BY cme_trade_date DESC;

,rollover_date,outgoing_contract,incoming_contract,new_order
0,2026-09-14,ESU6,ESZ6,"{ESZ6,ESU6}"
1,2026-06-15,ESM6,ESU6,"{ESU6,ESM6}"
2,2026-03-16,ESH6,ESM6,"{ESM6,ESH6}"
3,2025-12-15,ESZ5,ESH6,"{ESH6,ESZ5}"
4,2025-09-15,ESU5,ESZ5,"{ESZ5,ESU5}"
5,2025-06-16,ESM5,ESU5,"{ESU5,ESM5}"
6,2025-03-17,ESH5,ESM5,"{ESM5,ESH5}"
7,2024-12-16,ESZ4,ESH5,"{ESH5,ESZ4}"
8,2024-09-16,ESU4,ESZ4,"{ESZ4,ESU4}"
9,2024-06-17,ESM4,ESU4,"{ESU4,ESM4}"


DAY AFTER TRANSITION - USE OUTGOING FOR DAILY VARIANCE
Want to select from cme_es... the OUT contract name up to and incl the date, then IN.
Produce temp table w/ dates before and up to rollover w/ OUT next to it, then next with the next OUT?
Discard Holidays.

Find holidays

In [6]:
%%sql
SELECT DISTINCT
    MIN(cme_trade_date) AS earliest_date,
    MAX(cme_trade_date) AS latest_date
FROM
    cme_es_futures;

,earliest_date,latest_date
0,2023-10-02,2026-10-01


In [25]:
import pandas as pd
import pandas_market_calendars as mcal

cme = mcal.get_calendar('CME Globex Equity')
early = cme.schedule(start_date=start_fin["earliest_date"][0], end_date=start_fin["latest_date"][0])
early_close_dates = cme.early_closes(schedule=early)
print(early_close_dates.index)

DatetimeIndex(['2023-11-23', '2023-11-24', '2024-01-15', '2024-02-19',
               '2024-03-29', '2024-05-27', '2024-06-19', '2024-07-03',
               '2024-07-04', '2024-09-02', '2024-11-28', '2024-11-29',
               '2024-12-24', '2025-01-20', '2025-02-17', '2025-04-18',
               '2025-05-26', '2025-06-19', '2025-07-03', '2025-07-04',
               '2025-09-01', '2025-11-27', '2025-11-28', '2025-12-24',
               '2026-01-19', '2026-02-16', '2026-04-03', '2026-05-25',
               '2026-06-19', '2026-07-03', '2026-09-07'],
              dtype='datetime64[us]', freq=None)


In [ ]:
%%sql
